In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("p_environment", "dev", "environment")
v_env = dbutils.widgets.get("p_environment")

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-16", "file_date")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
from pyspark.sql.functions import col, lit, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

In [0]:
production_country_df = (
    spark.read
        .format("json")
        .option("multiline", "true")
        .load(f"{bronze_folder_path}/incremental_append/{v_file_date}/production_country")
    )
display(production_country_df)

In [0]:
production_country_shema = StructType([
    StructField("movieId", IntegerType()),
    StructField("countryId", IntegerType())
])

production_country_df = (
    spark.read
        .schema(production_country_shema)
        .format("json")
        .option("multiline", "true")
        .load(f"{bronze_folder_path}/incremental_append/{v_file_date}/production_country")
)
display(production_country_df)

In [0]:
production_country_df = (
    production_country_df
        .withColumnsRenamed(
            {
                "movieId": "movie_Id",
                "countryId": "country_id"
            }

        )
        .withColumns(
            {
                "env": lit(v_env),
                "file_date": lit(v_file_date)
            }
        )
)

movie_language_df = add_ingestion_date(production_country_df)

display(production_country_df)

In [0]:
overwrite_partitioned_table("movie_silver", "production_countries", "file_date", v_file_date)

In [0]:
(
    production_country_df
        .write
        .mode("append")
        .saveAsTable("movie_silver.production_countries")
)

In [0]:
dbutils.notebook.exit("Success")